<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GenericT_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score

# Reproducibility
def seed_everything(seed=42):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"]=str(seed); os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    np.random.seed(seed); random.seed(seed)

SEED=42
seed_everything(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)

# Load Hancock data
import gdown,json
gdown.download(id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",output="clinical_data.json",quiet=False)
gdown.download(id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",output="pathological_data.json",quiet=False)
with open("clinical_data.json","r") as f: clinical=pd.DataFrame(json.load(f))
with open("pathological_data.json","r") as f: pathology=pd.DataFrame(json.load(f))
df=clinical.merge(pathology,on="patient_id",how="inner")
print("Clinical shape:",clinical.shape)
print("Pathology shape:",pathology.shape)
print("Merged shape:",df.shape)

# HPV label
df=df[df["hpv_association_p16"].isin(["positive","negative"])].copy()
df["HPV"]=df["hpv_association_p16"].map({"negative":0,"positive":1})
print("\nHPV distribution:")
print(df["HPV"].value_counts())

# Hancock features
features=[
    "age_at_initial_diagnosis","sex","smoking_status","primarily_metastasis",
    "first_treatment_intent","first_treatment_modality","days_to_first_treatment",
    "adjuvant_treatment_intent","adjuvant_radiotherapy","adjuvant_radiotherapy_modality",
    "adjuvant_systemic_therapy","adjuvant_systemic_therapy_modality","adjuvant_radiochemotherapy",
    "primary_tumor_site","pT_stage","pN_stage","histologic_type",
    "number_of_positive_lymph_nodes","number_of_resected_lymph_nodes",
    "perinodal_invasion","lymphovascular_invasion_L","vascular_invasion_V",
    "perineural_invasion_Pn","resection_status","infiltration_depth_in_mm"
]
model_df=df[features+["HPV"]].copy()
X=model_df[features].copy()
y=model_df["HPV"].copy()

# Stratified train/test split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=SEED,stratify=y
)
print('train sample:',y_train.value_counts(),'\n')
print('test sample:',y_test.value_counts())

# Train-only preprocessing
categorical_features=[c for c in features if X_train[c].dtype==object]
numerical_features=[c for c in features if c not in categorical_features]

for col in categorical_features:
    value=X_train[col].mode().iloc[0] if not X_train[col].mode().empty else None
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

for col in numerical_features:
    value=X_train[col].median()
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

X_train=pd.get_dummies(X_train,columns=categorical_features,dtype=float)
X_test=pd.get_dummies(X_test,columns=categorical_features,dtype=float)
X_test=X_test.reindex(columns=X_train.columns,fill_value=0)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

# Torch conversion — no SMOTE
X_train=torch.FloatTensor(X_train)
y_train=torch.LongTensor(y_train.to_numpy())
X_test=torch.FloatTensor(X_test)
y_test=torch.LongTensor(y_test.to_numpy())

# Model Architecture
class HPVNet_Transformer(nn.Module):
    def __init__(self,num_features=78,d_model=32,nhead=4,num_layers=2,dim_feedforward=64,dropout=0.1,num_classes=2):
        super().__init__()
        self.num_features=num_features
        self.d_model=d_model
        self.feature_projection=nn.Linear(1,d_model)
        self.feature_embedding=nn.Parameter(torch.zeros(1,num_features,d_model))
        self.cls_token=nn.Parameter(torch.zeros(1,1,d_model))
        encoder_layer=nn.TransformerEncoderLayer(
            d_model=d_model,nhead=nhead,dim_feedforward=dim_feedforward,
            dropout=dropout,activation="gelu",batch_first=True,norm_first=True
        )
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.norm=nn.LayerNorm(d_model)
        self.classifier=nn.Sequential(
            nn.Linear(d_model,16),nn.ReLU(),nn.Dropout(dropout),nn.Linear(16,num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.feature_embedding,mean=0.0,std=0.02)
        nn.init.normal_(self.cls_token,mean=0.0,std=0.02)
        nn.init.xavier_uniform_(self.feature_projection.weight)
        nn.init.zeros_(self.feature_projection.bias)

    def forward(self,x):
        batch_size=x.size(0)
        x=self.feature_projection(x.unsqueeze(-1))
        x=x+self.feature_embedding
        cls_tokens=self.cls_token.expand(batch_size,-1,-1)
        x=torch.cat([cls_tokens,x],dim=1)
        x=self.transformer(x)
        x=self.norm(x[:,0])
        return self.classifier(x)

# Model, weights and optimisation
model=HPVNet_Transformer(
    num_features=X_train.shape[1],d_model=32,nhead=4,num_layers=2,
    dim_feedforward=64,dropout=0.1,num_classes=2
).to(device)

class_counts=np.bincount(y_train.numpy())
class_weights=len(y_train)/(len(class_counts)*class_counts)
weights=torch.tensor(class_weights,dtype=torch.float32,device=device)
criterion=nn.CrossEntropyLoss(weight=weights)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
train_loader=torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train,y_train),batch_size=256,shuffle=True
)

# Training — test-set checkpoint selection
epochs=500
best_loss=float("inf"); best_ba=-1; best_auc=-1
best_loss_epoch=best_ba_epoch=best_auc_epoch=0

for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad()
        train_loss=criterion(model(xb),yb)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs=model(X_test.to(device))
        test_loss=criterion(outputs,y_test.to(device)).item()
        probabilities=torch.softmax(outputs,dim=1)[:,1].cpu().numpy()
        predicted=(probabilities>=0.5).astype(int)

    test_ba=balanced_accuracy_score(y_test.numpy(),predicted)
    test_auc=roc_auc_score(y_test.numpy(),probabilities)

    if test_loss<best_loss:
        best_loss=test_loss; best_loss_epoch=epoch+1
        torch.save(model.state_dict(),"best_loss_transformer.pth")
    if test_ba>best_ba:
        best_ba=test_ba; best_ba_epoch=epoch+1
        torch.save(model.state_dict(),"best_ba_transformer.pth")
    if test_auc>best_auc:
        best_auc=test_auc; best_auc_epoch=epoch+1
        torch.save(model.state_dict(),"best_auc_transformer.pth")

    if (epoch+1)%50==0:
        print(f"Epoch {epoch+1}, Train={train_loss.item():.4f}, Test Loss={test_loss:.4f}, BA={test_ba:.4f}, AUC={test_auc:.4f}")

print(f"Best Test Loss={best_loss:.4f} at epoch {best_loss_epoch}")
print(f"Best Test BA={best_ba:.4f} at epoch {best_ba_epoch}")
print(f"Best Test AUC={best_auc:.4f} at epoch {best_auc_epoch}")

# Final evaluation
def evaluate_checkpoint(path):
    model.load_state_dict(torch.load(path,map_location=device))
    model.eval()
    with torch.no_grad():
        outputs=model(X_test.to(device))
        probabilities=torch.softmax(outputs,dim=1).cpu().numpy()
    predicted=np.argmax(probabilities,axis=1)
    true=y_test.numpy()
    print(f"\n{path}")
    print(classification_report(true,predicted,digits=4))
    print(f"Balanced Accuracy: {balanced_accuracy_score(true,predicted):.4f}")
    print(f"F1-score:          {f1_score(true,predicted):.4f}")
    print(f"AUC:               {roc_auc_score(true,probabilities[:,1]):.4f}")

evaluate_checkpoint("best_loss_transformer.pth")
evaluate_checkpoint("best_ba_transformer.pth")
evaluate_checkpoint("best_auc_transformer.pth")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 7.25MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 6.38MB/s]


Clinical shape: (763, 32)
Pathology shape: (763, 18)
Merged shape: (763, 49)

HPV distribution:
HPV
0    191
1    141
Name: count, dtype: int64
train sample: HPV
0    152
1    113
Name: count, dtype: int64 

test sample: HPV
0    39
1    28
Name: count, dtype: int64


/tmp/ipykernel_3264/1503445833.py:105: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 50, Train=0.6945, Test Loss=0.6915, BA=0.5000, AUC=0.7518
Epoch 100, Train=0.6403, Test Loss=0.5227, BA=0.7825, AUC=0.8617
Epoch 150, Train=0.3581, Test Loss=0.5219, BA=0.8082, AUC=0.8654
Epoch 200, Train=0.3457, Test Loss=0.4607, BA=0.8260, AUC=0.9048
Epoch 250, Train=0.2214, Test Loss=0.5844, BA=0.8104, AUC=0.8892
Epoch 300, Train=0.2058, Test Loss=0.4205, BA=0.8489, AUC=0.9185
Epoch 350, Train=0.4181, Test Loss=0.3985, BA=0.8489, AUC=0.9185
Epoch 400, Train=0.0710, Test Loss=0.5378, BA=0.8132, AUC=0.8535
Epoch 450, Train=0.4337, Test Loss=0.4515, BA=0.8210, AUC=0.8956
Epoch 500, Train=0.2564, Test Loss=0.3972, BA=0.8310, AUC=0.9029
Best Test Loss=0.3920 at epoch 272
Best Test BA=0.8695 at epoch 233
Best Test AUC=0.9277 at epoch 397

best_loss_transformer.pth
              precision    recall  f1-score   support

           0     0.9167    0.8462    0.8800        39
           1     0.8065    0.8929    0.8475        28

    accuracy                         0.8657        67
   m